In [ ]:
# =============================================================================
# CLO-SKET PARAMETER SENSITIVITY
# CELL 1 — LOAD FROZEN CANONICAL OBJECTS AND LOCK PRIMARY SETTINGS
# =============================================================================

import pickle
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 92)
print("CLO-SKET — PARAMETER SENSITIVITY")
print("CELL 1 — LOAD FROZEN CANONICAL OBJECTS")
print("=" * 92)

# -------------------------------------------------------------------------
# Frozen final object source
# -------------------------------------------------------------------------

OUT_DIR = Path("/content/drive/MyDrive/FashionAI")
PKL_PATH = OUT_DIR / "CLO_SKET_FINAL_IDENTITY_FIGURES.pkl"

if not PKL_PATH.exists():
    raise FileNotFoundError(
        f"Frozen final object file not found:\n{PKL_PATH}"
    )

with open(PKL_PATH, "rb") as f:
    frozen = pickle.load(f)

print(f"\nLoaded: {PKL_PATH}")
print(f"Objects available: {len(frozen)}")

# -------------------------------------------------------------------------
# Objects available from frozen final analysis
# -------------------------------------------------------------------------

required = [
    "garment_identity_ids",
    "cell30m_fold_assignment",
    "C2_obs",
    "S2_obs",
    "R2_obs",
    "mu2_obs_deg",
]

missing = [name for name in required if name not in frozen]

if missing:
    raise RuntimeError(
        "Frozen file missing required objects: "
        + ", ".join(missing)
    )

garment_identity_ids = np.asarray(
    frozen["garment_identity_ids"]
).copy()

cell30m_fold_assignment = np.asarray(
    frozen["cell30m_fold_assignment"]
).copy()

C2_obs = np.asarray(
    frozen["C2_obs"],
    dtype=float,
).copy()

S2_obs = np.asarray(
    frozen["S2_obs"],
    dtype=float,
).copy()

R2_obs = np.asarray(
    frozen["R2_obs"],
    dtype=float,
).copy()

mu2_obs_deg = np.asarray(
    frozen["mu2_obs_deg"],
    dtype=float,
).copy()

# -------------------------------------------------------------------------
# Canonical primary settings from audited final notebook
# -------------------------------------------------------------------------

PRIMARY_SETTINGS = {
    "n_angular_bins": 72,
    "n_radial_bins": 72,
    "locked_r_min": 3.5,
    "locked_r_max": 27.5,
    "support_threshold_fraction": 0.10,
    "concentration_half_width": 4.0,
    "harmonic_order": 2,
}

# Locked 25-shell coordinates
r_primary = np.arange(
    PRIMARY_SETTINGS["locked_r_min"],
    PRIMARY_SETTINGS["locked_r_max"] + 1e-12,
    1.0,
)

# -------------------------------------------------------------------------
# Structural audit
# -------------------------------------------------------------------------

assert garment_identity_ids.shape == (2300,)
assert cell30m_fold_assignment.shape == (2300,)

assert C2_obs.shape == (2300, 25)
assert S2_obs.shape == (2300, 25)
assert R2_obs.shape == (2300, 25)
assert mu2_obs_deg.shape == (2300, 25)

assert r_primary.shape == (25,)
assert np.isclose(r_primary[0], 3.5)
assert np.isclose(r_primary[-1], 27.5)

assert np.isfinite(C2_obs).all()
assert np.isfinite(S2_obs).all()
assert np.isfinite(R2_obs).all()
assert np.isfinite(mu2_obs_deg).all()

# Fundamental identity
R2_check = np.sqrt(
    C2_obs**2 + S2_obs**2
)

max_identity_error = float(
    np.max(
        np.abs(
            R2_check - R2_obs
        )
    )
)

assert max_identity_error < 1e-12

# -------------------------------------------------------------------------
# Report
# -------------------------------------------------------------------------

print("\nPRIMARY SETTINGS LOCK")
print("-" * 92)

for key, value in PRIMARY_SETTINGS.items():
    print(f"{key:32s}: {value}")

print("\nFROZEN DATA AUDIT")
print("-" * 92)

print(
    f"Sketches                         : "
    f"{len(garment_identity_ids)}"
)

print(
    f"Garment identities               : "
    f"{len(np.unique(garment_identity_ids))}"
)

print(
    f"Frozen folds                     : "
    f"{len(np.unique(cell30m_fold_assignment))}"
)

print(
    f"Observed field shape             : "
    f"{R2_obs.shape}"
)

print(
    f"Primary radial shells            : "
    f"{len(r_primary)}"
)

print(
    f"Primary radial domain            : "
    f"{r_primary.min():.1f} -> "
    f"{r_primary.max():.1f}"
)

print(
    f"max |R2 - sqrt(C2²+S2²)|         : "
    f"{max_identity_error:.3e}"
)

print("\nPASS — canonical primary settings and frozen objects loaded.")
print("No sensitivity perturbation has yet been applied.")

CLO-SKET — PARAMETER SENSITIVITY
CELL 1 — LOAD FROZEN CANONICAL OBJECTS

Loaded: /content/drive/MyDrive/FashionAI/CLO_SKET_FINAL_IDENTITY_FIGURES.pkl
Objects available: 35

PRIMARY SETTINGS LOCK
--------------------------------------------------------------------------------------------
n_angular_bins                  : 72
n_radial_bins                   : 72
locked_r_min                    : 3.5
locked_r_max                    : 27.5
support_threshold_fraction      : 0.1
concentration_half_width        : 4.0
harmonic_order                  : 2

FROZEN DATA AUDIT
--------------------------------------------------------------------------------------------
Sketches                         : 2300
Garment identities               : 230
Frozen folds                     : 5
Observed field shape             : (2300, 25)
Primary radial shells            : 25
Primary radial domain            : 3.5 -> 27.5
max |R2 - sqrt(C2²+S2²)|         : 0.000e+00

PASS — canonical primary settings and frozen

In [ ]:
# =============================================================================
# CLO-SKET PARAMETER SENSITIVITY
# CELL 2 — RADIAL DESCRIPTOR ENGINE + PRIMARY REPRODUCTION
# =============================================================================

import numpy as np
import pandas as pd

print("=" * 92)
print("CLO-SKET — PARAMETER SENSITIVITY")
print("CELL 2 — RADIAL DESCRIPTOR ENGINE + PRIMARY REPRODUCTION")
print("=" * 92)

# -------------------------------------------------------------------------
# Descriptor engine
#
# Reconstructs the eight frozen radial-magnitude descriptors:
#
#   1. integrated magnitude
#   2. magnitude-weighted radial centroid
#   3. magnitude-weighted radial spread
#   4. concentration around peak
#   5. onset radius
#   6. termination radius
#   7. peak radius
#   8. peak magnitude
#
# from an already-observed R2(r)=|F2(r)| field.
# -------------------------------------------------------------------------

def compute_radial_descriptors(
    R_field,
    r_values,
    support_threshold_fraction=0.10,
    concentration_half_width=4.0,
):
    """
    Compute the eight radial descriptors used in the final 14-D representation.

    Parameters
    ----------
    R_field : ndarray, shape (n_sketches, n_shells)
        Observed second-harmonic magnitude field.

    r_values : ndarray, shape (n_shells,)
        Radial shell centers.

    support_threshold_fraction : float
        Threshold fraction tau = fraction * peak magnitude.

    concentration_half_width : float
        Half-width in radial-coordinate units around the discrete peak.

    Returns
    -------
    descriptors : ndarray, shape (n_sketches, 8)

    names : list[str]
    """

    R_field = np.asarray(R_field, dtype=float)
    r_values = np.asarray(r_values, dtype=float)

    if R_field.ndim != 2:
        raise ValueError("R_field must be 2-D.")

    if r_values.ndim != 1:
        raise ValueError("r_values must be 1-D.")

    if R_field.shape[1] != len(r_values):
        raise ValueError(
            "R_field shell dimension must match r_values."
        )

    if not (0.0 < support_threshold_fraction < 1.0):
        raise ValueError(
            "support_threshold_fraction must lie in (0,1)."
        )

    if concentration_half_width <= 0:
        raise ValueError(
            "concentration_half_width must be positive."
        )

    n = R_field.shape[0]

    descriptors = np.full(
        (n, 8),
        np.nan,
        dtype=float,
    )

    for i in range(n):

        m = R_field[i]

        # -------------------------------------------------------------
        # Integrated magnitude
        # -------------------------------------------------------------

        I = float(
            np.trapz(
                m,
                x=r_values,
            )
        )

        if I <= 0:
            raise RuntimeError(
                f"Sketch {i}: non-positive integrated magnitude."
            )

        # -------------------------------------------------------------
        # Magnitude-weighted radial centroid
        # -------------------------------------------------------------

        r_bar = float(
            np.trapz(
                r_values * m,
                x=r_values,
            )
            / I
        )

        # -------------------------------------------------------------
        # Magnitude-weighted radial spread
        # -------------------------------------------------------------

        variance = float(
            np.trapz(
                ((r_values - r_bar) ** 2) * m,
                x=r_values,
            )
            / I
        )

        variance = max(
            variance,
            0.0,
        )

        spread = float(
            np.sqrt(variance)
        )

        # -------------------------------------------------------------
        # Discrete observed peak
        # -------------------------------------------------------------

        peak_idx = int(
            np.argmax(m)
        )

        peak_radius = float(
            r_values[peak_idx]
        )

        peak_magnitude = float(
            m[peak_idx]
        )

        # -------------------------------------------------------------
        # Concentration around peak
        # -------------------------------------------------------------

        concentration_mask = (
            (r_values >= peak_radius - concentration_half_width)
            &
            (r_values <= peak_radius + concentration_half_width)
        )

        concentration_integral = float(
            np.trapz(
                m[concentration_mask],
                x=r_values[concentration_mask],
            )
        )

        concentration = (
            concentration_integral / I
        )

        # -------------------------------------------------------------
        # Threshold-defined support
        # -------------------------------------------------------------

        tau = (
            support_threshold_fraction
            * peak_magnitude
        )

        support_mask = (
            m >= tau
        )

        if not np.any(support_mask):
            raise RuntimeError(
                f"Sketch {i}: no support shells found."
            )

        onset_radius = float(
            r_values[
                np.where(support_mask)[0][0]
            ]
        )

        termination_radius = float(
            r_values[
                np.where(support_mask)[0][-1]
            ]
        )

        descriptors[i] = [
            I,
            r_bar,
            spread,
            concentration,
            onset_radius,
            termination_radius,
            peak_radius,
            peak_magnitude,
        ]

    names = [
        "integrated_magnitude",
        "radial_centroid",
        "radial_spread",
        "peak_concentration",
        "onset_radius",
        "termination_radius",
        "peak_radius",
        "peak_magnitude",
    ]

    assert np.isfinite(descriptors).all()

    return descriptors, names


# -------------------------------------------------------------------------
# Primary reconstruction
# -------------------------------------------------------------------------

radial_primary_8d, radial_feature_names = (
    compute_radial_descriptors(
        R_field=R2_obs,
        r_values=r_primary,
        support_threshold_fraction=(
            PRIMARY_SETTINGS[
                "support_threshold_fraction"
            ]
        ),
        concentration_half_width=(
            PRIMARY_SETTINGS[
                "concentration_half_width"
            ]
        ),
    )
)

assert radial_primary_8d.shape == (
    2300,
    8,
)

# -------------------------------------------------------------------------
# Basic descriptive audit
# -------------------------------------------------------------------------

primary_summary = pd.DataFrame({
    "feature": radial_feature_names,
    "min": np.min(
        radial_primary_8d,
        axis=0,
    ),
    "median": np.median(
        radial_primary_8d,
        axis=0,
    ),
    "mean": np.mean(
        radial_primary_8d,
        axis=0,
    ),
    "max": np.max(
        radial_primary_8d,
        axis=0,
    ),
})

print("\nPRIMARY 8-D RADIAL DESCRIPTOR SUMMARY")
print("-" * 92)

print(
    primary_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

# -------------------------------------------------------------------------
# Structural checks
# -------------------------------------------------------------------------

peak_radius_primary = radial_primary_8d[:, 6]
peak_magnitude_primary = radial_primary_8d[:, 7]

direct_peak_idx = np.argmax(
    R2_obs,
    axis=1,
)

direct_peak_radius = r_primary[
    direct_peak_idx
]

direct_peak_magnitude = R2_obs[
    np.arange(2300),
    direct_peak_idx,
]

max_peak_radius_error = float(
    np.max(
        np.abs(
            peak_radius_primary
            - direct_peak_radius
        )
    )
)

max_peak_magnitude_error = float(
    np.max(
        np.abs(
            peak_magnitude_primary
            - direct_peak_magnitude
        )
    )
)

print("\nPRIMARY REPRODUCTION AUDIT")
print("-" * 92)

print(
    f"max |derived peak radius - direct|    : "
    f"{max_peak_radius_error:.3e}"
)

print(
    f"max |derived peak magnitude - direct| : "
    f"{max_peak_magnitude_error:.3e}"
)

assert max_peak_radius_error == 0.0
assert max_peak_magnitude_error == 0.0

print("\nPASS — radial descriptor engine reproduces the primary peak quantities.")
print(
    "The frozen 25-shell field is now ready for "
    "threshold, concentration-width, and domain sensitivity."
)

CLO-SKET — PARAMETER SENSITIVITY
CELL 2 — RADIAL DESCRIPTOR ENGINE + PRIMARY REPRODUCTION

PRIMARY 8-D RADIAL DESCRIPTOR SUMMARY
--------------------------------------------------------------------------------------------
             feature       min    median      mean       max
integrated_magnitude  1.688234  7.891117  8.021782 15.652514
     radial_centroid  8.865539 15.959676 15.780756 22.721959
       radial_spread  2.988923  7.007863  6.898863  8.627931
  peak_concentration  0.080703  0.365544  0.363261  0.837350
        onset_radius  3.500000  3.500000  3.971739 22.500000
  termination_radius 15.500000 27.500000 27.031304 27.500000
         peak_radius  3.500000 13.500000 14.924783 27.500000
      peak_magnitude  0.214069  0.660428  0.644050  0.890940

PRIMARY REPRODUCTION AUDIT
--------------------------------------------------------------------------------------------
max |derived peak radius - direct|    : 0.000e+00
max |derived peak magnitude - direct| : 0.000e+00

PASS — 

/tmp/ipykernel_3668/1784196596.py:102: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  np.trapz(
/tmp/ipykernel_3668/1784196596.py:118: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  np.trapz(
/tmp/ipykernel_3668/1784196596.py:130: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  np.trapz(
/tmp/ipykernel_3668/1784196596.py:173: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  np.trapz(


In [ ]:
# =============================================================================
# CLO-SKET PARAMETER SENSITIVITY
# CELL 3 — DEFINE AND COMPUTE PRESPECIFIED SENSITIVITY GRID
# =============================================================================

import numpy as np
import pandas as pd

print("=" * 92)
print("CLO-SKET — PARAMETER SENSITIVITY")
print("CELL 3 — DEFINE AND COMPUTE PRESPECIFIED SENSITIVITY GRID")
print("=" * 92)

# -------------------------------------------------------------------------
# Updated descriptor engine using np.trapezoid
# -------------------------------------------------------------------------

def compute_radial_descriptors_v2(
    R_field,
    r_values,
    support_threshold_fraction=0.10,
    concentration_half_width=4.0,
):
    R_field = np.asarray(R_field, dtype=float)
    r_values = np.asarray(r_values, dtype=float)

    n = R_field.shape[0]
    descriptors = np.full((n, 8), np.nan, dtype=float)

    for i in range(n):

        m = R_field[i]

        I = float(np.trapezoid(m, x=r_values))

        r_bar = float(
            np.trapezoid(r_values * m, x=r_values) / I
        )

        variance = float(
            np.trapezoid(
                ((r_values - r_bar) ** 2) * m,
                x=r_values,
            ) / I
        )

        spread = float(np.sqrt(max(variance, 0.0)))

        peak_idx = int(np.argmax(m))
        peak_radius = float(r_values[peak_idx])
        peak_magnitude = float(m[peak_idx])

        concentration_mask = (
            (r_values >= peak_radius - concentration_half_width)
            &
            (r_values <= peak_radius + concentration_half_width)
        )

        concentration = float(
            np.trapezoid(
                m[concentration_mask],
                x=r_values[concentration_mask],
            ) / I
        )

        tau = support_threshold_fraction * peak_magnitude
        support_mask = m >= tau

        onset_radius = float(
            r_values[np.where(support_mask)[0][0]]
        )

        termination_radius = float(
            r_values[np.where(support_mask)[0][-1]]
        )

        descriptors[i] = [
            I,
            r_bar,
            spread,
            concentration,
            onset_radius,
            termination_radius,
            peak_radius,
            peak_magnitude,
        ]

    return descriptors


# -------------------------------------------------------------------------
# Prespecified sensitivity values
# -------------------------------------------------------------------------

THRESHOLDS = [0.05, 0.10, 0.15]

CONCENTRATION_WIDTHS = [2.0, 4.0, 6.0]

DOMAINS = [
    (3.5, 27.5),
    (4.5, 26.5),
    (5.5, 25.5),
]

print("\nSENSITIVITY GRID")
print("-" * 92)
print(f"Support thresholds       : {THRESHOLDS}")
print(f"Concentration half-widths: {CONCENTRATION_WIDTHS}")
print(f"Radial domains           : {DOMAINS}")

# -------------------------------------------------------------------------
# Helper to subset frozen field to a narrower radial domain
# -------------------------------------------------------------------------

def subset_domain(R_field, r_values, r_min, r_max):

    mask = (
        (r_values >= r_min)
        &
        (r_values <= r_max)
    )

    return (
        R_field[:, mask],
        r_values[mask],
    )


# -------------------------------------------------------------------------
# Compute one-at-a-time sensitivity variants
#
# Each family changes ONE primary design choice at a time.
# -------------------------------------------------------------------------

sensitivity_objects = {}
sensitivity_rows = []

# -------------------------------------------------------------------------
# A. Threshold sensitivity
# -------------------------------------------------------------------------

for tau in THRESHOLDS:

    desc = compute_radial_descriptors_v2(
        R2_obs,
        r_primary,
        support_threshold_fraction=tau,
        concentration_half_width=4.0,
    )

    key = f"threshold_{tau:.2f}"

    sensitivity_objects[key] = desc

    sensitivity_rows.append({
        "family": "support_threshold",
        "setting": tau,
        "domain_min": 3.5,
        "domain_max": 27.5,
        "n_shells": 25,
    })


# -------------------------------------------------------------------------
# B. Concentration-width sensitivity
# -------------------------------------------------------------------------

for width in CONCENTRATION_WIDTHS:

    desc = compute_radial_descriptors_v2(
        R2_obs,
        r_primary,
        support_threshold_fraction=0.10,
        concentration_half_width=width,
    )

    key = f"concentration_{width:.1f}"

    sensitivity_objects[key] = desc

    sensitivity_rows.append({
        "family": "concentration_width",
        "setting": width,
        "domain_min": 3.5,
        "domain_max": 27.5,
        "n_shells": 25,
    })


# -------------------------------------------------------------------------
# C. Radial-domain sensitivity
# -------------------------------------------------------------------------

for r_min, r_max in DOMAINS:

    R_sub, r_sub = subset_domain(
        R2_obs,
        r_primary,
        r_min,
        r_max,
    )

    desc = compute_radial_descriptors_v2(
        R_sub,
        r_sub,
        support_threshold_fraction=0.10,
        concentration_half_width=4.0,
    )

    key = f"domain_{r_min:.1f}_{r_max:.1f}"

    sensitivity_objects[key] = desc

    sensitivity_rows.append({
        "family": "radial_domain",
        "setting": f"{r_min:.1f}-{r_max:.1f}",
        "domain_min": r_min,
        "domain_max": r_max,
        "n_shells": len(r_sub),
    })


# -------------------------------------------------------------------------
# Primary configuration from updated engine
# -------------------------------------------------------------------------

primary_v2 = compute_radial_descriptors_v2(
    R2_obs,
    r_primary,
    support_threshold_fraction=0.10,
    concentration_half_width=4.0,
)

# Exact agreement with Cell 2
max_primary_engine_difference = float(
    np.max(
        np.abs(
            primary_v2 - radial_primary_8d
        )
    )
)

assert max_primary_engine_difference < 1e-12


# -------------------------------------------------------------------------
# Configuration audit
# -------------------------------------------------------------------------

sensitivity_design_df = pd.DataFrame(
    sensitivity_rows
)

print("\nSENSITIVITY DESIGN TABLE")
print("-" * 92)

print(
    sensitivity_design_df.to_string(
        index=False
    )
)

print("\nENGINE REPRODUCTION")
print("-" * 92)
print(
    "max |updated engine - Cell 2 primary| : "
    f"{max_primary_engine_difference:.3e}"
)

# -------------------------------------------------------------------------
# Hard structural guards
# -------------------------------------------------------------------------

for key, arr in sensitivity_objects.items():

    assert arr.shape == (2300, 8)
    assert np.isfinite(arr).all()

print("\nPASS — all one-at-a-time sensitivity variants computed.")
print(
    "Primary configuration remains unchanged and reproduces exactly."
)

CLO-SKET — PARAMETER SENSITIVITY
CELL 3 — DEFINE AND COMPUTE PRESPECIFIED SENSITIVITY GRID

SENSITIVITY GRID
--------------------------------------------------------------------------------------------
Support thresholds       : [0.05, 0.1, 0.15]
Concentration half-widths: [2.0, 4.0, 6.0]
Radial domains           : [(3.5, 27.5), (4.5, 26.5), (5.5, 25.5)]

SENSITIVITY DESIGN TABLE
--------------------------------------------------------------------------------------------
             family  setting  domain_min  domain_max  n_shells
  support_threshold     0.05         3.5        27.5        25
  support_threshold      0.1         3.5        27.5        25
  support_threshold     0.15         3.5        27.5        25
concentration_width      2.0         3.5        27.5        25
concentration_width      4.0         3.5        27.5        25
concentration_width      6.0         3.5        27.5        25
      radial_domain 3.5-27.5         3.5        27.5        25
      radial_domain 

In [ ]:
# =============================================================================
# CLO-SKET PARAMETER SENSITIVITY
# CELL 4 — FEATURE-BY-FEATURE SENSITIVITY QUANTIFICATION
# =============================================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

print("=" * 92)
print("CLO-SKET — PARAMETER SENSITIVITY")
print("CELL 4 — FEATURE-BY-FEATURE SENSITIVITY QUANTIFICATION")
print("=" * 92)

primary = primary_v2.copy()

feature_names = radial_feature_names

# -------------------------------------------------------------------------
# Helper
# -------------------------------------------------------------------------

def summarize_variant(
    variant_key,
    variant_array,
    primary_array,
    feature_names,
):
    rows = []

    for j, feature in enumerate(feature_names):

        x = primary_array[:, j]
        y = variant_array[:, j]

        rho = float(
            spearmanr(x, y).statistic
        )

        abs_change = np.abs(y - x)

        median_abs_change = float(
            np.median(abs_change)
        )

        mean_abs_change = float(
            np.mean(abs_change)
        )

        max_abs_change = float(
            np.max(abs_change)
        )

        unchanged_prop = float(
            np.mean(
                np.isclose(
                    x,
                    y,
                    rtol=0.0,
                    atol=1e-12,
                )
            )
        )

        rows.append({
            "variant": variant_key,
            "feature": feature,
            "spearman_rho": rho,
            "median_abs_change": median_abs_change,
            "mean_abs_change": mean_abs_change,
            "max_abs_change": max_abs_change,
            "unchanged_prop": unchanged_prop,
        })

    return rows


# -------------------------------------------------------------------------
# Compute all comparisons
# -------------------------------------------------------------------------

comparison_rows = []

for key, arr in sensitivity_objects.items():

    comparison_rows.extend(
        summarize_variant(
            variant_key=key,
            variant_array=arr,
            primary_array=primary,
            feature_names=feature_names,
        )
    )

sensitivity_feature_df = pd.DataFrame(
    comparison_rows
)

# -------------------------------------------------------------------------
# Add family label
# -------------------------------------------------------------------------

def variant_family(key):

    if key.startswith("threshold_"):
        return "support_threshold"

    if key.startswith("concentration_"):
        return "concentration_width"

    if key.startswith("domain_"):
        return "radial_domain"

    return "unknown"


sensitivity_feature_df["family"] = (
    sensitivity_feature_df[
        "variant"
    ].map(variant_family)
)

# -------------------------------------------------------------------------
# Identify exact primary duplicates
# -------------------------------------------------------------------------

PRIMARY_VARIANTS = {
    "threshold_0.10",
    "concentration_4.0",
    "domain_3.5_27.5",
}

sensitivity_feature_df[
    "is_primary_setting"
] = (
    sensitivity_feature_df[
        "variant"
    ].isin(PRIMARY_VARIANTS)
)

# -------------------------------------------------------------------------
# Full table excluding duplicate primary rows
# -------------------------------------------------------------------------

nonprimary_df = sensitivity_feature_df[
    ~sensitivity_feature_df[
        "is_primary_setting"
    ]
].copy()

print("\nNON-PRIMARY FEATURE SENSITIVITY")
print("-" * 92)

print(
    nonprimary_df[
        [
            "family",
            "variant",
            "feature",
            "spearman_rho",
            "median_abs_change",
            "mean_abs_change",
            "max_abs_change",
            "unchanged_prop",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

# -------------------------------------------------------------------------
# Compact worst-case summary per feature
# -------------------------------------------------------------------------

worst_case_rows = []

for feature in feature_names:

    sub = nonprimary_df[
        nonprimary_df["feature"] == feature
    ]

    worst_case_rows.append({
        "feature": feature,
        "min_spearman_rho": float(
            sub["spearman_rho"].min()
        ),
        "max_median_abs_change": float(
            sub["median_abs_change"].max()
        ),
        "max_mean_abs_change": float(
            sub["mean_abs_change"].max()
        ),
        "max_abs_change": float(
            sub["max_abs_change"].max()
        ),
        "min_unchanged_prop": float(
            sub["unchanged_prop"].min()
        ),
    })

worst_case_df = pd.DataFrame(
    worst_case_rows
)

print("\nWORST-CASE SENSITIVITY BY FEATURE")
print("-" * 92)

print(
    worst_case_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

# -------------------------------------------------------------------------
# Exact shell-agreement summaries
#
# These are especially interpretable for discrete radial features.
# -------------------------------------------------------------------------

discrete_features = [
    "onset_radius",
    "termination_radius",
    "peak_radius",
]

print("\nDISCRETE RADIAL FEATURE AGREEMENT")
print("-" * 92)

discrete_rows = []

for key, arr in sensitivity_objects.items():

    if key in PRIMARY_VARIANTS:
        continue

    for feature in discrete_features:

        j = feature_names.index(
            feature
        )

        exact_match = float(
            np.mean(
                arr[:, j]
                ==
                primary[:, j]
            )
        )

        within_1_shell = float(
            np.mean(
                np.abs(
                    arr[:, j]
                    -
                    primary[:, j]
                )
                <= 1.0
            )
        )

        within_2_shells = float(
            np.mean(
                np.abs(
                    arr[:, j]
                    -
                    primary[:, j]
                )
                <= 2.0
            )
        )

        discrete_rows.append({
            "variant": key,
            "feature": feature,
            "exact_match_prop": exact_match,
            "within_1_shell_prop": within_1_shell,
            "within_2_shells_prop": within_2_shells,
        })

discrete_df = pd.DataFrame(
    discrete_rows
)

print(
    discrete_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

# -------------------------------------------------------------------------
# Boundary occupancy audit under domain variants
# -------------------------------------------------------------------------

print("\nDOMAIN-BOUNDARY PEAK OCCUPANCY")
print("-" * 92)

domain_boundary_rows = []

for r_min, r_max in DOMAINS:

    key = f"domain_{r_min:.1f}_{r_max:.1f}"

    arr = sensitivity_objects[key]

    peak_r = arr[:, 6]

    lower_prop = float(
        np.mean(
            peak_r == r_min
        )
    )

    upper_prop = float(
        np.mean(
            peak_r == r_max
        )
    )

    endpoint_prop = float(
        np.mean(
            (peak_r == r_min)
            |
            (peak_r == r_max)
        )
    )

    domain_boundary_rows.append({
        "domain": f"{r_min:.1f}-{r_max:.1f}",
        "lower_endpoint_prop": lower_prop,
        "upper_endpoint_prop": upper_prop,
        "either_endpoint_prop": endpoint_prop,
    })

domain_boundary_df = pd.DataFrame(
    domain_boundary_rows
)

print(
    domain_boundary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print("\nPASS — feature-level parameter sensitivity quantified.")

CLO-SKET — PARAMETER SENSITIVITY
CELL 4 — FEATURE-BY-FEATURE SENSITIVITY QUANTIFICATION

NON-PRIMARY FEATURE SENSITIVITY
--------------------------------------------------------------------------------------------
             family           variant              feature  spearman_rho  median_abs_change  mean_abs_change  max_abs_change  unchanged_prop
  support_threshold    threshold_0.05 integrated_magnitude      1.000000           0.000000         0.000000        0.000000        1.000000
  support_threshold    threshold_0.05      radial_centroid      1.000000           0.000000         0.000000        0.000000        1.000000
  support_threshold    threshold_0.05        radial_spread      1.000000           0.000000         0.000000        0.000000        1.000000
  support_threshold    threshold_0.05   peak_concentration      1.000000           0.000000         0.000000        0.000000        1.000000
  support_threshold    threshold_0.05         onset_radius      0.875220         

In [ ]:
# =============================================================================
# CLO-SKET PARAMETER SENSITIVITY
# CELL 5 — DISCOVER CANONICAL FULL-FIELD RUNTIME SOURCE
# =============================================================================

import pickle
import numpy as np
from pathlib import Path

print("=" * 92)
print("CLO-SKET — PARAMETER SENSITIVITY")
print("CELL 5 — DISCOVER CANONICAL FULL-FIELD RUNTIME SOURCE")
print("=" * 92)

ROOT = Path("/content/drive/MyDrive/FashionAI")

# -------------------------------------------------------------------------
# Candidate runtime backups known from the audited analysis lineage
# -------------------------------------------------------------------------

candidate_paths = [
    ROOT / "CLO_SKET_runtime_backup.pkl",
    ROOT / "CLO_SKET_runtime_backup_AFTER_CELL25.pkl",
]

required_full_objects = [
    "radial_centers",
    "F2_mag",
]

optional_upstream_objects = [
    "conditional_angular",
    "conditional_fft",
    "conditional_fourier",
]

discovery_rows = []

for path in candidate_paths:

    print("\n" + "-" * 92)
    print(f"Checking: {path}")

    if not path.exists():

        print("STATUS: FILE NOT FOUND")

        discovery_rows.append({
            "path": str(path),
            "exists": False,
            "object_count": np.nan,
            "has_radial_centers": False,
            "has_F2_mag": False,
            "has_conditional_angular": False,
            "has_conditional_fft": False,
            "has_conditional_fourier": False,
        })

        continue

    try:

        with open(path, "rb") as f:
            obj = pickle.load(f)

    except Exception as exc:

        print(
            "STATUS: LOAD FAILED\n"
            f"{type(exc).__name__}: {exc}"
        )

        continue

    if not isinstance(obj, dict):

        print(
            "STATUS: loaded object is not a dictionary:"
            f" {type(obj)}"
        )

        continue

    keys = set(obj.keys())

    print(f"STATUS: LOADED")
    print(f"Objects: {len(keys)}")

    for name in (
        required_full_objects
        + optional_upstream_objects
    ):

        if name in obj:

            arr = np.asarray(obj[name])

            print(
                f"  {name:24s} PRESENT "
                f"shape={arr.shape} "
                f"dtype={arr.dtype}"
            )

        else:

            print(
                f"  {name:24s} MISSING"
            )

    discovery_rows.append({
        "path": str(path),
        "exists": True,
        "object_count": len(keys),
        "has_radial_centers":
            "radial_centers" in keys,
        "has_F2_mag":
            "F2_mag" in keys,
        "has_conditional_angular":
            "conditional_angular" in keys,
        "has_conditional_fft":
            "conditional_fft" in keys,
        "has_conditional_fourier":
            "conditional_fourier" in keys,
    })

print("\n" + "=" * 92)
print("DISCOVERY COMPLETE")
print("=" * 92)

for row in discovery_rows:

    print()
    for key, value in row.items():
        print(f"{key:28s}: {value}")

print(
    "\nNo sensitivity calculation has been performed in this cell."
)
print(
    "This cell only identifies the authoritative saved full-field source."
)

CLO-SKET — PARAMETER SENSITIVITY
CELL 5 — DISCOVER CANONICAL FULL-FIELD RUNTIME SOURCE

--------------------------------------------------------------------------------------------
Checking: /content/drive/MyDrive/FashionAI/CLO_SKET_runtime_backup.pkl
STATUS: FILE NOT FOUND

--------------------------------------------------------------------------------------------
Checking: /content/drive/MyDrive/FashionAI/CLO_SKET_runtime_backup_AFTER_CELL25.pkl
STATUS: FILE NOT FOUND

DISCOVERY COMPLETE

path                        : /content/drive/MyDrive/FashionAI/CLO_SKET_runtime_backup.pkl
exists                      : False
object_count                : nan
has_radial_centers          : False
has_F2_mag                  : False
has_conditional_angular     : False
has_conditional_fft         : False
has_conditional_fourier     : False

path                        : /content/drive/MyDrive/FashionAI/CLO_SKET_runtime_backup_AFTER_CELL25.pkl
exists                      : False
object_count         

In [ ]:
# =============================================================================
# CLO-SKET PARAMETER SENSITIVITY
# CELL 6 — RECOMPUTE CANONICAL FULL 72×72 RADIAL–ANGULAR FIELD
# =============================================================================

import os
import numpy as np
from pathlib import Path
from PIL import Image

print("=" * 92)
print("CLO-SKET — PARAMETER SENSITIVITY")
print("CELL 6 — RECOMPUTE CANONICAL FULL 72×72 FIELD")
print("=" * 92)

# -------------------------------------------------------------------------
# Exact canonical source and discretization from audited notebook
# -------------------------------------------------------------------------

DATA_ROOT = Path(
    "/content/drive/MyDrive/FashionAI/datasets/Clo-Sket/Clo-Sket"
)

N_ANGULAR = 72
N_RADIAL = 72

MASS_EPS = 1e-14
MASS_TOL = 1e-10

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Dataset root does not exist:\n{DATA_ROOT}"
    )

# -------------------------------------------------------------------------
# 1. Exact source inventory
# -------------------------------------------------------------------------

image_paths = []
image_categories = []

for category in sorted(os.listdir(DATA_ROOT)):

    category_path = DATA_ROOT / category

    if not category_path.is_dir():
        continue

    for filename in sorted(os.listdir(category_path)):

        if filename.lower().endswith(
            (".tif", ".tiff")
        ):
            image_paths.append(
                str(category_path / filename)
            )
            image_categories.append(category)

image_paths = np.asarray(image_paths)
image_categories = np.asarray(image_categories)

N = len(image_paths)

print("\nSOURCE")
print("-" * 92)
print(f"Dataset root        : {DATA_ROOT}")
print(f"Images              : {N}")
print(f"Categories          : {len(np.unique(image_categories))}")
print(f"Angular bins        : {N_ANGULAR}")
print(f"Radial bins         : {N_RADIAL}")

assert N == 2300
assert len(np.unique(image_categories)) == 23

# -------------------------------------------------------------------------
# 2. Polar-bin geometry
# -------------------------------------------------------------------------

theta_edges = np.linspace(
    -np.pi,
    np.pi,
    N_ANGULAR + 1,
)

theta_centers = (
    theta_edges[:-1]
    + theta_edges[1:]
) / 2.0

normalized_r_edges = np.linspace(
    0.0,
    1.0,
    N_RADIAL + 1,
)

normalized_r_centers = (
    normalized_r_edges[:-1]
    + normalized_r_edges[1:]
) / 2.0

# Downstream descriptor coordinate used in audited notebook:
# 0.5, 1.5, ..., 71.5
radial_centers_full = (
    np.arange(
        N_RADIAL,
        dtype=float,
    )
    + 0.5
)

assert np.allclose(
    radial_centers_full,
    normalized_r_centers * N_RADIAL,
    atol=1e-12,
    rtol=0.0,
)

# -------------------------------------------------------------------------
# 3. Storage
# -------------------------------------------------------------------------

joint_mass = np.zeros(
    (
        N,
        N_RADIAL,
        N_ANGULAR,
    ),
    dtype=np.float64,
)

centroids = np.zeros(
    (N, 2),
    dtype=np.float64,
)

total_mass = np.zeros(
    N,
    dtype=np.float64,
)

# -------------------------------------------------------------------------
# 4. Recover exact radial × angular ink mass
# -------------------------------------------------------------------------

print("\nRECOVERING RAW RADIAL × ANGULAR MASS")
print("-" * 92)

for i, path in enumerate(image_paths):

    with Image.open(path) as im:

        im.load()

        img = np.asarray(
            im.convert("L"),
            dtype=np.float64,
        )

    if img.ndim != 2:
        raise RuntimeError(
            f"Expected grayscale image: {path}"
        )

    if not np.isfinite(img).all():
        raise RuntimeError(
            f"Non-finite pixels: {path}"
        )

    # Continuous darkness / ink weight.
    w = np.maximum(
        255.0 - img,
        0.0,
    )

    mass = float(np.sum(w))

    if not np.isfinite(mass) or mass <= 0:
        raise RuntimeError(
            f"Invalid ink mass: {path}"
        )

    total_mass[i] = mass

    height, width = w.shape

    scale = float(
        max(width, height)
    )

    x = (
        np.arange(
            width,
            dtype=np.float64,
        )
        - (width - 1) / 2.0
    ) / scale

    y = (
        np.arange(
            height,
            dtype=np.float64,
        )
        - (height - 1) / 2.0
    ) / scale

    X, Y = np.meshgrid(
        x,
        y,
    )

    # Intensity-weighted centroid.
    cx = float(
        np.sum(w * X) / mass
    )

    cy = float(
        np.sum(w * Y) / mass
    )

    centroids[i] = [cx, cy]

    Xc = X - cx
    Yc = Y - cy

    radius = np.sqrt(
        Xc**2 + Yc**2
    )

    theta = np.arctan2(
        Yc,
        Xc,
    )

    # Normalize radius by the largest available radius in this image,
    # matching the audited Cell 11R lineage.
    r_max = float(
        np.max(radius)
    )

    if r_max <= 0:
        raise RuntimeError(
            f"Invalid radial maximum: {path}"
        )

    radius_norm = radius / r_max

    # Weighted 2-D histogram.
    hist, _, _ = np.histogram2d(
        radius_norm.ravel(),
        theta.ravel(),
        bins=[
            normalized_r_edges,
            theta_edges,
        ],
        weights=w.ravel(),
    )

    joint_mass[i] = hist

    if (i + 1) % 250 == 0 or (i + 1) == N:
        print(
            f"Processed {i + 1:4d}/{N}"
        )

# -------------------------------------------------------------------------
# 5. Mass-conservation audit
# -------------------------------------------------------------------------

recovered_mass = np.sum(
    joint_mass,
    axis=(1, 2),
)

relative_mass_error = np.abs(
    recovered_mass - total_mass
) / total_mass

max_relative_mass_error = float(
    np.max(relative_mass_error)
)

print("\nMASS CONSERVATION")
print("-" * 92)
print(
    f"max relative mass error : "
    f"{max_relative_mass_error:.3e}"
)

assert max_relative_mass_error < MASS_TOL

# -------------------------------------------------------------------------
# 6. Conditional angular distribution p(theta | r)
# -------------------------------------------------------------------------

radial_mass_full = np.sum(
    joint_mass,
    axis=2,
)

conditional_angular_full = np.zeros_like(
    joint_mass,
    dtype=np.float64,
)

positive_shell = (
    radial_mass_full > MASS_EPS
)

conditional_angular_full[
    positive_shell
] = (
    joint_mass[positive_shell]
    /
    radial_mass_full[
        positive_shell,
        None,
    ]
)

# -------------------------------------------------------------------------
# 7. Conditional angular Fourier transform
#
# np.fft.rfft uses exp(-i k theta), matching the audited convention.
# -------------------------------------------------------------------------

conditional_fft_full = np.fft.rfft(
    conditional_angular_full,
    axis=2,
)

F2_full = conditional_fft_full[:, :, 2]

C2_full = np.real(
    F2_full
)

S2_full = -np.imag(
    F2_full
)

R2_full = np.abs(
    F2_full
)

mu2_full_deg = np.mod(
    0.5
    * np.degrees(
        np.arctan2(
            S2_full,
            C2_full,
        )
    ),
    180.0,
)

# -------------------------------------------------------------------------
# 8. Full-field structural audit
# -------------------------------------------------------------------------

assert conditional_angular_full.shape == (
    2300,
    72,
    72,
)

assert F2_full.shape == (
    2300,
    72,
)

assert C2_full.shape == (
    2300,
    72,
)

assert S2_full.shape == (
    2300,
    72,
)

assert R2_full.shape == (
    2300,
    72,
)

assert radial_centers_full.shape == (
    72,
)

R_identity_error = float(
    np.max(
        np.abs(
            R2_full
            -
            np.sqrt(
                C2_full**2
                + S2_full**2
            )
        )
    )
)

assert R_identity_error < 1e-12

# -------------------------------------------------------------------------
# 9. Crucial reproduction check against frozen primary 25-shell field
# -------------------------------------------------------------------------

primary_mask_full = (
    (radial_centers_full >= 3.5)
    &
    (radial_centers_full <= 27.5)
)

R2_primary_from_full = R2_full[
    :,
    primary_mask_full,
]

C2_primary_from_full = C2_full[
    :,
    primary_mask_full,
]

S2_primary_from_full = S2_full[
    :,
    primary_mask_full,
]

print("\nPRIMARY 25-SHELL REPRODUCTION")
print("-" * 92)

print(
    "Recovered primary shape  : "
    f"{R2_primary_from_full.shape}"
)

max_R_primary_diff = float(
    np.max(
        np.abs(
            R2_primary_from_full
            - R2_obs
        )
    )
)

max_C_primary_diff = float(
    np.max(
        np.abs(
            C2_primary_from_full
            - C2_obs
        )
    )
)

max_S_primary_diff = float(
    np.max(
        np.abs(
            S2_primary_from_full
            - S2_obs
        )
    )
)

print(
    f"max |R2_full-lock - frozen R2| : "
    f"{max_R_primary_diff:.3e}"
)

print(
    f"max |C2_full-lock - frozen C2| : "
    f"{max_C_primary_diff:.3e}"
)

print(
    f"max |S2_full-lock - frozen S2| : "
    f"{max_S_primary_diff:.3e}"
)

# Hard guard: this raw re-extraction must reproduce the frozen
# final field before it is allowed into sensitivity analysis.
assert max_R_primary_diff < 1e-10
assert max_C_primary_diff < 1e-10
assert max_S_primary_diff < 1e-10

# -------------------------------------------------------------------------
# 10. Save canonical upstream sensitivity source
# -------------------------------------------------------------------------

FULL_FIELD_PATH = (
    OUT_DIR
    / "CLO_SKET_PARAMETER_SENSITIVITY_FULL72.npz"
)

np.savez_compressed(
    FULL_FIELD_PATH,
    radial_centers_full=radial_centers_full,
    C2_full=C2_full,
    S2_full=S2_full,
    R2_full=R2_full,
    mu2_full_deg=mu2_full_deg,
    radial_mass_full=radial_mass_full,
)

print("\nFULL FIELD")
print("-" * 92)
print(
    f"Radial extent : "
    f"{radial_centers_full.min():.1f} "
    f"-> "
    f"{radial_centers_full.max():.1f}"
)
print(
    f"Saved         : {FULL_FIELD_PATH}"
)

print("\nPASS — canonical full 72-shell field reproduced from raw TIFFs.")
print(
    "The frozen primary 25-shell field was reproduced before "
    "the full field was accepted for sensitivity analysis."
)

CLO-SKET — PARAMETER SENSITIVITY
CELL 6 — RECOMPUTE CANONICAL FULL 72×72 FIELD

SOURCE
--------------------------------------------------------------------------------------------
Dataset root        : /content/drive/MyDrive/FashionAI/datasets/Clo-Sket/Clo-Sket
Images              : 2300
Categories          : 23
Angular bins        : 72
Radial bins         : 72

RECOVERING RAW RADIAL × ANGULAR MASS
--------------------------------------------------------------------------------------------
Processed  250/2300
Processed  500/2300
Processed  750/2300
Processed 1000/2300
Processed 1250/2300
Processed 1500/2300
Processed 1750/2300
Processed 2000/2300
Processed 2250/2300
Processed 2300/2300

MASS CONSERVATION
--------------------------------------------------------------------------------------------
max relative mass error : 0.000e+00

PRIMARY 25-SHELL REPRODUCTION
--------------------------------------------------------------------------------------------
Recovered primary shape  : (2300,

In [ ]:
# =============================================================================
# CLO-SKET PARAMETER SENSITIVITY
# CELL 7 — FULL-FIELD RADIAL-DOMAIN EXPANSION SENSITIVITY
# =============================================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

print("=" * 92)
print("CLO-SKET — PARAMETER SENSITIVITY")
print("CELL 7 — FULL-FIELD RADIAL-DOMAIN EXPANSION SENSITIVITY")
print("=" * 92)

# -------------------------------------------------------------------------
# Domains
#
# Primary:
#   [3.5, 27.5]
#
# Outward symmetric expansions:
#   [2.5, 28.5]
#   [1.5, 29.5]
#   [0.5, 30.5]
#
# Existing inward trims retained for comparison:
#   [4.5, 26.5]
#   [5.5, 25.5]
#
# -------------------------------------------------------------------------

FULL_DOMAIN_SENSITIVITY = [
    (5.5, 25.5, "inward_2"),
    (4.5, 26.5, "inward_1"),
    (3.5, 27.5, "primary"),
    (2.5, 28.5, "outward_1"),
    (1.5, 29.5, "outward_2"),
    (0.5, 30.5, "outward_3"),
]

# -------------------------------------------------------------------------
# Helper
# -------------------------------------------------------------------------

def get_domain_descriptor(
    R_full,
    r_full,
    r_min,
    r_max,
):

    mask = (
        (r_full >= r_min)
        &
        (r_full <= r_max)
    )

    R_sub = R_full[:, mask]
    r_sub = r_full[mask]

    desc = compute_radial_descriptors_v2(
        R_sub,
        r_sub,
        support_threshold_fraction=0.10,
        concentration_half_width=4.0,
    )

    return desc, r_sub


# -------------------------------------------------------------------------
# Primary reference from full field
# -------------------------------------------------------------------------

primary_full_desc, primary_full_r = (
    get_domain_descriptor(
        R2_full,
        radial_centers_full,
        3.5,
        27.5,
    )
)

# Must exactly reproduce current primary descriptor engine.
max_primary_descriptor_diff = float(
    np.max(
        np.abs(
            primary_full_desc
            -
            primary_v2
        )
    )
)

assert max_primary_descriptor_diff < 1e-12

# -------------------------------------------------------------------------
# Compute all domains
# -------------------------------------------------------------------------

domain_objects_full = {}
domain_summary_rows = []
domain_feature_rows = []

for r_min, r_max, label in FULL_DOMAIN_SENSITIVITY:

    desc, r_sub = get_domain_descriptor(
        R2_full,
        radial_centers_full,
        r_min,
        r_max,
    )

    domain_objects_full[label] = {
        "descriptors": desc,
        "r_values": r_sub,
        "r_min": r_min,
        "r_max": r_max,
    }

    peak_r = desc[:, 6]
    peak_mag = desc[:, 7]

    primary_peak_r = primary_full_desc[:, 6]
    primary_peak_mag = primary_full_desc[:, 7]

    # ---------------------------------------------------------------------
    # Endpoint occupancy
    # ---------------------------------------------------------------------

    lower_prop = float(
        np.mean(
            peak_r == r_min
        )
    )

    upper_prop = float(
        np.mean(
            peak_r == r_max
        )
    )

    endpoint_prop = float(
        np.mean(
            (peak_r == r_min)
            |
            (peak_r == r_max)
        )
    )

    # ---------------------------------------------------------------------
    # Peak migration relative to primary
    # ---------------------------------------------------------------------

    exact_peak_match = float(
        np.mean(
            peak_r
            ==
            primary_peak_r
        )
    )

    peak_shift = (
        peak_r
        -
        primary_peak_r
    )

    median_abs_peak_shift = float(
        np.median(
            np.abs(
                peak_shift
            )
        )
    )

    mean_abs_peak_shift = float(
        np.mean(
            np.abs(
                peak_shift
            )
        )
    )

    max_abs_peak_shift = float(
        np.max(
            np.abs(
                peak_shift
            )
        )
    )

    # Direction of migration
    prop_peak_moves_outward = float(
        np.mean(
            peak_r
            >
            primary_peak_r
        )
    )

    prop_peak_moves_inward = float(
        np.mean(
            peak_r
            <
            primary_peak_r
        )
    )

    # ---------------------------------------------------------------------
    # Peak-magnitude change
    # ---------------------------------------------------------------------

    peak_mag_delta = (
        peak_mag
        -
        primary_peak_mag
    )

    median_peak_mag_delta = float(
        np.median(
            peak_mag_delta
        )
    )

    mean_peak_mag_delta = float(
        np.mean(
            peak_mag_delta
        )
    )

    # ---------------------------------------------------------------------
    # Peak-rank stability
    # ---------------------------------------------------------------------

    peak_radius_rho = float(
        spearmanr(
            primary_peak_r,
            peak_r,
        ).statistic
    )

    peak_magnitude_rho = float(
        spearmanr(
            primary_peak_mag,
            peak_mag,
        ).statistic
    )

    domain_summary_rows.append({
        "label": label,
        "domain": f"{r_min:.1f}-{r_max:.1f}",
        "n_shells": len(r_sub),
        "lower_endpoint_prop":
            lower_prop,
        "upper_endpoint_prop":
            upper_prop,
        "either_endpoint_prop":
            endpoint_prop,
        "exact_peak_match_prop":
            exact_peak_match,
        "median_abs_peak_shift":
            median_abs_peak_shift,
        "mean_abs_peak_shift":
            mean_abs_peak_shift,
        "max_abs_peak_shift":
            max_abs_peak_shift,
        "prop_peak_moves_outward":
            prop_peak_moves_outward,
        "prop_peak_moves_inward":
            prop_peak_moves_inward,
        "peak_radius_spearman":
            peak_radius_rho,
        "peak_magnitude_spearman":
            peak_magnitude_rho,
        "median_peak_magnitude_delta":
            median_peak_mag_delta,
        "mean_peak_magnitude_delta":
            mean_peak_mag_delta,
    })

    # ---------------------------------------------------------------------
    # All eight descriptor correlations with primary
    # ---------------------------------------------------------------------

    for j, feature in enumerate(
        radial_feature_names
    ):

        x = primary_full_desc[:, j]
        y = desc[:, j]

        rho = float(
            spearmanr(
                x,
                y,
            ).statistic
        )

        abs_change = np.abs(
            y - x
        )

        exact_match = float(
            np.mean(
                np.isclose(
                    x,
                    y,
                    atol=1e-12,
                    rtol=0.0,
                )
            )
        )

        domain_feature_rows.append({
            "label": label,
            "domain": f"{r_min:.1f}-{r_max:.1f}",
            "feature": feature,
            "spearman_rho": rho,
            "median_abs_change":
                float(np.median(abs_change)),
            "mean_abs_change":
                float(np.mean(abs_change)),
            "exact_match_prop":
                exact_match,
        })

# -------------------------------------------------------------------------
# Results
# -------------------------------------------------------------------------

full_domain_summary_df = pd.DataFrame(
    domain_summary_rows
)

full_domain_feature_df = pd.DataFrame(
    domain_feature_rows
)

print("\nDOMAIN-LEVEL PEAK SENSITIVITY")
print("-" * 92)

print(
    full_domain_summary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print("\nDESCRIPTOR RANK STABILITY BY DOMAIN")
print("-" * 92)

rank_table = (
    full_domain_feature_df
    .pivot(
        index="feature",
        columns="label",
        values="spearman_rho",
    )
)

# Use intuitive column order
desired_cols = [
    "inward_2",
    "inward_1",
    "primary",
    "outward_1",
    "outward_2",
    "outward_3",
]

rank_table = rank_table[
    desired_cols
]

print(
    rank_table.to_string(
        float_format=lambda x: f"{x:.6f}",
    )
)

# -------------------------------------------------------------------------
# Specifically audit sketches whose PRIMARY peak sat at upper boundary
# -------------------------------------------------------------------------

primary_peak_r = primary_full_desc[:, 6]

primary_upper_mask = (
    primary_peak_r == 27.5
)

n_primary_upper = int(
    np.sum(
        primary_upper_mask
    )
)

print("\nPRIMARY UPPER-BOUNDARY PEAK AUDIT")
print("-" * 92)
print(
    f"Primary peaks at 27.5 : "
    f"{n_primary_upper}/{len(primary_peak_r)} "
    f"({np.mean(primary_upper_mask):.6f})"
)

for label in [
    "outward_1",
    "outward_2",
    "outward_3",
]:

    desc = domain_objects_full[
        label
    ]["descriptors"]

    peak_r_expanded = desc[:, 6]

    vals = peak_r_expanded[
        primary_upper_mask
    ]

    remained_27_5 = float(
        np.mean(
            vals == 27.5
        )
    )

    moved_beyond_27_5 = float(
        np.mean(
            vals > 27.5
        )
    )

    median_new_peak = float(
        np.median(
            vals
        )
    )

    max_new_peak = float(
        np.max(
            vals
        )
    )

    print(
        f"{label:12s} | "
        f"remain 27.5={remained_27_5:.6f} | "
        f"move >27.5={moved_beyond_27_5:.6f} | "
        f"median new peak={median_new_peak:.2f} | "
        f"max new peak={max_new_peak:.2f}"
    )

# -------------------------------------------------------------------------
# Hard guards
# -------------------------------------------------------------------------

primary_row = full_domain_summary_df[
    full_domain_summary_df[
        "label"
    ] == "primary"
].iloc[0]

assert (
    primary_row[
        "exact_peak_match_prop"
    ]
    == 1.0
)

assert (
    abs(
        primary_row[
            "either_endpoint_prop"
        ]
        - 0.22043478260869565
    )
    < 1e-12
)

print("\nPASS — full-field radial-domain sensitivity quantified.")
print(
    "The primary window can now be evaluated against genuine outward expansions."
)

CLO-SKET — PARAMETER SENSITIVITY
CELL 7 — FULL-FIELD RADIAL-DOMAIN EXPANSION SENSITIVITY

DOMAIN-LEVEL PEAK SENSITIVITY
--------------------------------------------------------------------------------------------
    label   domain  n_shells  lower_endpoint_prop  upper_endpoint_prop  either_endpoint_prop  exact_peak_match_prop  median_abs_peak_shift  mean_abs_peak_shift  max_abs_peak_shift  prop_peak_moves_outward  prop_peak_moves_inward  peak_radius_spearman  peak_magnitude_spearman  median_peak_magnitude_delta  mean_peak_magnitude_delta
 inward_2 5.5-25.5        21             0.127826             0.132609              0.260435               0.635217               0.000000             3.125217           22.000000                 0.202609                0.162174              0.593675                 0.885816                     0.000000                  -0.028091
 inward_1 4.5-26.5        23             0.125652             0.117391              0.243043               0.779565        

In [ ]:
# =============================================================================
# CLO-SKET PARAMETER SENSITIVITY
# CELL 8 — ANGULAR-RESOLUTION SENSITIVITY BY EXACT MASS AGGREGATION
# =============================================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

print("=" * 92)
print("CLO-SKET — PARAMETER SENSITIVITY")
print("CELL 8 — ANGULAR-RESOLUTION SENSITIVITY")
print("=" * 92)

# -------------------------------------------------------------------------
# Canonical source
# -------------------------------------------------------------------------
#
# joint_mass shape:
#     sketch × radial_bin × angular_bin
#
# Canonical angular resolution = 72.
#
# We test exact divisors of 72:
#     36 bins -> aggregate pairs
#     24 bins -> aggregate triplets
#
# No interpolation or image reprocessing is involved.
# -------------------------------------------------------------------------

assert joint_mass.shape == (2300, 72, 72)

ANGULAR_RESOLUTIONS = [24, 36, 72]

# -------------------------------------------------------------------------
# Helper: exact angular aggregation
# -------------------------------------------------------------------------

def aggregate_angular_mass(joint_mass_72, n_angular_new):

    joint_mass_72 = np.asarray(
        joint_mass_72,
        dtype=float,
    )

    n_old = joint_mass_72.shape[2]

    if n_old % n_angular_new != 0:
        raise ValueError(
            f"{n_angular_new} does not divide {n_old} exactly."
        )

    group_size = n_old // n_angular_new

    reshaped = joint_mass_72.reshape(
        joint_mass_72.shape[0],
        joint_mass_72.shape[1],
        n_angular_new,
        group_size,
    )

    aggregated = reshaped.sum(
        axis=3
    )

    return aggregated


# -------------------------------------------------------------------------
# Helper: compute F2 field from an angular mass tensor
# -------------------------------------------------------------------------

def harmonic_field_from_mass(
    angular_mass,
    n_angular,
    harmonic_order=2,
):

    angular_mass = np.asarray(
        angular_mass,
        dtype=float,
    )

    radial_mass = angular_mass.sum(
        axis=2
    )

    conditional = np.zeros_like(
        angular_mass,
        dtype=float,
    )

    positive = radial_mass > 1e-14

    conditional[positive] = (
        angular_mass[positive]
        /
        radial_mass[positive, None]
    )

    fft = np.fft.rfft(
        conditional,
        axis=2,
    )

    F = fft[:, :, harmonic_order]

    C = np.real(F)
    S = -np.imag(F)
    R = np.abs(F)

    mu_deg = np.mod(
        0.5
        * np.degrees(
            np.arctan2(
                S,
                C,
            )
        ),
        180.0,
    )

    return {
        "conditional": conditional,
        "C2": C,
        "S2": S,
        "R2": R,
        "mu2_deg": mu_deg,
    }


# -------------------------------------------------------------------------
# Compute all angular resolutions
# -------------------------------------------------------------------------

angular_results = {}
angular_summary_rows = []

for n_ang in ANGULAR_RESOLUTIONS:

    print(f"\nComputing angular resolution: {n_ang}")

    if n_ang == 72:
        mass_new = joint_mass.copy()
    else:
        mass_new = aggregate_angular_mass(
            joint_mass,
            n_ang,
        )

    result = harmonic_field_from_mass(
        mass_new,
        n_angular=n_ang,
        harmonic_order=2,
    )

    angular_results[n_ang] = result

    # -------------------------------------------------------------
    # Restrict to the canonical primary radial domain
    # -------------------------------------------------------------

    primary_mask = (
        (radial_centers_full >= 3.5)
        &
        (radial_centers_full <= 27.5)
    )

    R_new = result["R2"][
        :,
        primary_mask,
    ]

    C_new = result["C2"][
        :,
        primary_mask,
    ]

    S_new = result["S2"][
        :,
        primary_mask,
    ]

    mu_new = result["mu2_deg"][
        :,
        primary_mask,
    ]

    # -------------------------------------------------------------
    # Compare with canonical 72-bin primary field
    # -------------------------------------------------------------

    R_rho = float(
        spearmanr(
            R2_obs.ravel(),
            R_new.ravel(),
        ).statistic
    )

    C_rho = float(
        spearmanr(
            C2_obs.ravel(),
            C_new.ravel(),
        ).statistic
    )

    S_rho = float(
        spearmanr(
            S2_obs.ravel(),
            S_new.ravel(),
        ).statistic
    )

    R_mae = float(
        np.mean(
            np.abs(
                R_new - R2_obs
            )
        )
    )

    C_mae = float(
        np.mean(
            np.abs(
                C_new - C2_obs
            )
        )
    )

    S_mae = float(
        np.mean(
            np.abs(
                S_new - S2_obs
            )
        )
    )

    axial_err = np.mod(
        np.abs(
            mu_new - mu2_obs_deg
        ),
        180.0,
    )

    axial_err = np.minimum(
        axial_err,
        180.0 - axial_err,
    )

    median_axial_diff = float(
        np.median(
            axial_err
        )
    )

    mean_axial_diff = float(
        np.mean(
            axial_err
        )
    )

    # -------------------------------------------------------------
    # Peak quantities under canonical radial window
    # -------------------------------------------------------------

    peak_idx_new = np.argmax(
        R_new,
        axis=1,
    )

    peak_r_new = r_primary[
        peak_idx_new
    ]

    peak_mag_new = R_new[
        np.arange(2300),
        peak_idx_new,
    ]

    primary_peak_idx = np.argmax(
        R2_obs,
        axis=1,
    )

    primary_peak_r = r_primary[
        primary_peak_idx
    ]

    primary_peak_mag = R2_obs[
        np.arange(2300),
        primary_peak_idx,
    ]

    peak_radius_match = float(
        np.mean(
            peak_r_new
            ==
            primary_peak_r
        )
    )

    peak_radius_rho = float(
        spearmanr(
            primary_peak_r,
            peak_r_new,
        ).statistic
    )

    peak_magnitude_rho = float(
        spearmanr(
            primary_peak_mag,
            peak_mag_new,
        ).statistic
    )

    angular_summary_rows.append({
        "n_angular_bins": n_ang,
        "R2_spearman": R_rho,
        "C2_spearman": C_rho,
        "S2_spearman": S_rho,
        "R2_MAE_vs_72": R_mae,
        "C2_MAE_vs_72": C_mae,
        "S2_MAE_vs_72": S_mae,
        "median_axial_diff_deg":
            median_axial_diff,
        "mean_axial_diff_deg":
            mean_axial_diff,
        "peak_radius_exact_match":
            peak_radius_match,
        "peak_radius_spearman":
            peak_radius_rho,
        "peak_magnitude_spearman":
            peak_magnitude_rho,
    })

# -------------------------------------------------------------------------
# Results table
# -------------------------------------------------------------------------

angular_resolution_df = pd.DataFrame(
    angular_summary_rows
)

print("\nANGULAR-RESOLUTION SENSITIVITY")
print("-" * 92)

print(
    angular_resolution_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

# -------------------------------------------------------------------------
# Exact 72-bin reproduction guard
# -------------------------------------------------------------------------

row72 = angular_resolution_df[
    angular_resolution_df[
        "n_angular_bins"
    ] == 72
].iloc[0]

assert abs(
    row72["R2_spearman"] - 1.0
) < 1e-12

assert row72["R2_MAE_vs_72"] < 1e-12
assert row72["C2_MAE_vs_72"] < 1e-12
assert row72["S2_MAE_vs_72"] < 1e-12

assert (
    row72["median_axial_diff_deg"]
    < 1e-10
)

assert (
    row72["peak_radius_exact_match"]
    == 1.0
)

print("\nPASS — angular-resolution sensitivity quantified.")
print(
    "24- and 36-bin fields were derived by exact aggregation "
    "of the canonical 72-bin mass tensor."
)

CLO-SKET — PARAMETER SENSITIVITY
CELL 8 — ANGULAR-RESOLUTION SENSITIVITY

Computing angular resolution: 24

Computing angular resolution: 36

Computing angular resolution: 72

ANGULAR-RESOLUTION SENSITIVITY
--------------------------------------------------------------------------------------------
 n_angular_bins  R2_spearman  C2_spearman  S2_spearman  R2_MAE_vs_72  C2_MAE_vs_72  S2_MAE_vs_72  median_axial_diff_deg  mean_axial_diff_deg  peak_radius_exact_match  peak_radius_spearman  peak_magnitude_spearman
             24     0.997051     0.995460     0.912654      0.011560      0.023836      0.052468               5.039957             5.410261                 0.862174              0.952519                 0.994252
             36     0.999193     0.998844     0.971118      0.005910      0.011378      0.026790               2.529985             2.731462                 0.926522              0.973262                 0.998305
             72     1.000000     1.000000     1.000000      0

In [ ]:
# =============================================================================
# CLO-SKET PARAMETER SENSITIVITY
# CELL 9 — RADIAL-RESOLUTION SENSITIVITY BY EXACT MASS AGGREGATION
# =============================================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

print("=" * 92)
print("CLO-SKET — PARAMETER SENSITIVITY")
print("CELL 9 — RADIAL-RESOLUTION SENSITIVITY")
print("=" * 92)

assert joint_mass.shape == (2300, 72, 72)

RADIAL_RESOLUTIONS = [24, 36, 72]

# -------------------------------------------------------------------------
# IMPORTANT:
#
# Radial coordinate is expressed in normalized physical radius [0,1].
#
# Canonical 72-bin shell centers:
#     (j + 0.5) / 72
#
# The primary domain 3.5 ... 27.5 therefore corresponds to:
#
#     3/72 <= normalized radius < 28/72
#
# i.e. exact BIN EDGES, not approximate center matching.
#
# This allows exact corresponding domains at 36 and 24 bins:
#
# 72 bins : bins 3 ... 27  -> 25 shells
# 36 bins : bins ?         -> exact grouped equivalent where possible
# 24 bins : bins ?         -> exact grouped equivalent where possible
#
# Because the primary boundaries do not align perfectly with every
# coarser grouping, we compare over the nearest fully-contained
# common physical interval rather than silently changing the domain.
# -------------------------------------------------------------------------

PRIMARY_LOW_EDGE = 3.0 / 72.0
PRIMARY_HIGH_EDGE = 28.0 / 72.0

print("\nPRIMARY NORMALIZED RADIAL DOMAIN")
print("-" * 92)
print(f"Lower edge : {PRIMARY_LOW_EDGE:.9f}")
print(f"Upper edge : {PRIMARY_HIGH_EDGE:.9f}")

# -------------------------------------------------------------------------
# Exact radial aggregation
# -------------------------------------------------------------------------

def aggregate_radial_mass(
    joint_mass_72,
    n_radial_new,
):

    joint_mass_72 = np.asarray(
        joint_mass_72,
        dtype=float,
    )

    n_old = joint_mass_72.shape[1]

    if n_old % n_radial_new != 0:
        raise ValueError(
            f"{n_radial_new} does not divide {n_old} exactly."
        )

    group_size = n_old // n_radial_new

    reshaped = joint_mass_72.reshape(
        joint_mass_72.shape[0],
        n_radial_new,
        group_size,
        joint_mass_72.shape[2],
    )

    aggregated = reshaped.sum(
        axis=2
    )

    return aggregated


# -------------------------------------------------------------------------
# F2 from radial × angular mass
# -------------------------------------------------------------------------

def radial_harmonic_from_mass(
    mass,
):

    radial_mass = mass.sum(
        axis=2
    )

    conditional = np.zeros_like(
        mass,
        dtype=float,
    )

    positive = radial_mass > 1e-14

    conditional[positive] = (
        mass[positive]
        /
        radial_mass[positive, None]
    )

    fft = np.fft.rfft(
        conditional,
        axis=2,
    )

    F2 = fft[:, :, 2]

    C2 = np.real(F2)
    S2 = -np.imag(F2)
    R2 = np.abs(F2)

    mu2 = np.mod(
        0.5
        * np.degrees(
            np.arctan2(
                S2,
                C2,
            )
        ),
        180.0,
    )

    return C2, S2, R2, mu2


# -------------------------------------------------------------------------
# Compute resolutions
# -------------------------------------------------------------------------

radial_resolution_objects = {}

for n_rad in RADIAL_RESOLUTIONS:

    print(f"\nComputing radial resolution: {n_rad}")

    if n_rad == 72:
        mass_new = joint_mass.copy()
    else:
        mass_new = aggregate_radial_mass(
            joint_mass,
            n_rad,
        )

    C, S, R, mu = radial_harmonic_from_mass(
        mass_new
    )

    edges = np.linspace(
        0.0,
        1.0,
        n_rad + 1,
    )

    centers = (
        edges[:-1] + edges[1:]
    ) / 2.0

    radial_resolution_objects[n_rad] = {
        "mass": mass_new,
        "C2": C,
        "S2": S,
        "R2": R,
        "mu2_deg": mu,
        "edges": edges,
        "centers": centers,
    }


# -------------------------------------------------------------------------
# Determine COMMON interval represented completely at all resolutions.
#
# A coarse radial bin is retained only if its complete interval lies
# inside the canonical primary physical domain.
# -------------------------------------------------------------------------

for n_rad in RADIAL_RESOLUTIONS:

    obj = radial_resolution_objects[n_rad]

    edges = obj["edges"]

    lower_edges = edges[:-1]
    upper_edges = edges[1:]

    contained = (
        (lower_edges >= PRIMARY_LOW_EDGE - 1e-15)
        &
        (upper_edges <= PRIMARY_HIGH_EDGE + 1e-15)
    )

    obj["primary_contained_mask"] = contained

    print(
        f"{n_rad:2d} bins | "
        f"fully-contained primary shells = "
        f"{contained.sum()}"
    )


# -------------------------------------------------------------------------
# Descriptor calculation in normalized physical radius
# -------------------------------------------------------------------------

radial_descriptor_objects = {}

for n_rad in RADIAL_RESOLUTIONS:

    obj = radial_resolution_objects[n_rad]

    mask = obj[
        "primary_contained_mask"
    ]

    R_sub = obj["R2"][:, mask]

    r_sub = obj["centers"][mask]

    # Concentration half-width:
    #
    # canonical +/-4 shell-coordinate units corresponds physically to
    # +/-4/72 of normalized radius.
    physical_half_width = 4.0 / 72.0

    desc = compute_radial_descriptors_v2(
        R_sub,
        r_sub,
        support_threshold_fraction=0.10,
        concentration_half_width=physical_half_width,
    )

    radial_descriptor_objects[n_rad] = {
        "descriptors": desc,
        "r_values": r_sub,
    }


# -------------------------------------------------------------------------
# Use 72-bin result as reference
# -------------------------------------------------------------------------

ref = radial_descriptor_objects[
    72
]["descriptors"]

resolution_rows = []

for n_rad in RADIAL_RESOLUTIONS:

    desc = radial_descriptor_objects[
        n_rad
    ]["descriptors"]

    for j, feature in enumerate(
        radial_feature_names
    ):

        x = ref[:, j]
        y = desc[:, j]

        rho = float(
            spearmanr(
                x,
                y,
            ).statistic
        )

        resolution_rows.append({
            "n_radial_bins": n_rad,
            "feature": feature,
            "spearman_vs_72": rho,
            "median_abs_change":
                float(
                    np.median(
                        np.abs(y - x)
                    )
                ),
            "mean_abs_change":
                float(
                    np.mean(
                        np.abs(y - x)
                    )
                ),
        })

radial_resolution_df = pd.DataFrame(
    resolution_rows
)

print("\nRADIAL-RESOLUTION DESCRIPTOR STABILITY")
print("-" * 92)

rho_table = radial_resolution_df.pivot(
    index="feature",
    columns="n_radial_bins",
    values="spearman_vs_72",
)

print(
    rho_table.to_string(
        float_format=lambda x: f"{x:.6f}",
    )
)


# -------------------------------------------------------------------------
# Peak-location comparison in normalized physical coordinates
# -------------------------------------------------------------------------

ref_peak = ref[:, 6]

print("\nPEAK-RADIUS STABILITY")
print("-" * 92)

for n_rad in RADIAL_RESOLUTIONS:

    peak = radial_descriptor_objects[
        n_rad
    ]["descriptors"][:, 6]

    rho = float(
        spearmanr(
            ref_peak,
            peak,
        ).statistic
    )

    median_abs = float(
        np.median(
            np.abs(
                peak - ref_peak
            )
        )
    )

    mean_abs = float(
        np.mean(
            np.abs(
                peak - ref_peak
            )
        )
    )

    print(
        f"{n_rad:2d} bins | "
        f"rho={rho:.6f} | "
        f"median |delta r|={median_abs:.6f} | "
        f"mean |delta r|={mean_abs:.6f}"
    )


# -------------------------------------------------------------------------
# Hard 72-bin internal reproduction
# -------------------------------------------------------------------------

row72 = radial_resolution_df[
    radial_resolution_df[
        "n_radial_bins"
    ] == 72
]

assert np.allclose(
    row72["spearman_vs_72"],
    1.0,
    atol=1e-12,
)

print("\nPASS — radial-resolution sensitivity quantified.")
print(
    "Coarser fields were produced by exact aggregation of canonical "
    "radial mass; comparisons use normalized physical radius."
)

CLO-SKET — PARAMETER SENSITIVITY
CELL 9 — RADIAL-RESOLUTION SENSITIVITY

PRIMARY NORMALIZED RADIAL DOMAIN
--------------------------------------------------------------------------------------------
Lower edge : 0.041666667
Upper edge : 0.388888889

Computing radial resolution: 24

Computing radial resolution: 36

Computing radial resolution: 72
24 bins | fully-contained primary shells = 8
36 bins | fully-contained primary shells = 12
72 bins | fully-contained primary shells = 25

RADIAL-RESOLUTION DESCRIPTOR STABILITY
--------------------------------------------------------------------------------------------
n_radial_bins              24       36       72
feature                                        
integrated_magnitude 0.949575 0.975532 1.000000
onset_radius         0.616922 0.611106 1.000000
peak_concentration   0.551185 0.550322 1.000000
peak_magnitude       0.867258 0.898314 1.000000
peak_radius          0.644193 0.678076 1.000000
radial_centroid      0.914832 0.950406 1.00000

In [ ]:
# =============================================================================
# CLO-SKET PARAMETER SENSITIVITY
# CELL 10 — RADIAL-RESOLUTION CONTROL ON EXACT COMMON PHYSICAL DOMAIN
# =============================================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

print("=" * 92)
print("CLO-SKET — PARAMETER SENSITIVITY")
print("CELL 10 — RADIAL RESOLUTION ON EXACT COMMON PHYSICAL DOMAIN")
print("=" * 92)

# -------------------------------------------------------------------------
# Exact common boundaries across 24, 36 and 72 radial grids
#
# gcd(24,36,72) = 12
#
# Largest useful common interval fully inside the original primary domain:
#
#     [1/12, 1/3]
#
# 72 bins -> 18 shells
# 36 bins ->  9 shells
# 24 bins ->  6 shells
# -------------------------------------------------------------------------

COMMON_LOW = 1.0 / 12.0
COMMON_HIGH = 1.0 / 3.0

print("\nCOMMON PHYSICAL DOMAIN")
print("-" * 92)
print(f"Lower edge : {COMMON_LOW:.9f}")
print(f"Upper edge : {COMMON_HIGH:.9f}")

common_descriptor_objects = {}
common_domain_rows = []

# -------------------------------------------------------------------------
# Extract exactly corresponding physical intervals
# -------------------------------------------------------------------------

for n_rad in RADIAL_RESOLUTIONS:

    obj = radial_resolution_objects[n_rad]

    edges = obj["edges"]
    centers = obj["centers"]

    lower_edges = edges[:-1]
    upper_edges = edges[1:]

    mask = (
        (lower_edges >= COMMON_LOW - 1e-15)
        &
        (upper_edges <= COMMON_HIGH + 1e-15)
    )

    # Exact boundary checks
    selected_idx = np.where(mask)[0]

    first_lower = float(
        lower_edges[selected_idx[0]]
    )

    last_upper = float(
        upper_edges[selected_idx[-1]]
    )

    assert abs(first_lower - COMMON_LOW) < 1e-12
    assert abs(last_upper - COMMON_HIGH) < 1e-12

    R_sub = obj["R2"][:, mask]
    r_sub = centers[mask]

    # Same physical concentration half-width as canonical +/-4/72
    physical_half_width = 4.0 / 72.0

    desc = compute_radial_descriptors_v2(
        R_sub,
        r_sub,
        support_threshold_fraction=0.10,
        concentration_half_width=physical_half_width,
    )

    common_descriptor_objects[n_rad] = {
        "descriptors": desc,
        "r_values": r_sub,
        "mask": mask,
    }

    common_domain_rows.append({
        "n_radial_bins": n_rad,
        "n_selected_shells": int(mask.sum()),
        "first_lower_edge": first_lower,
        "last_upper_edge": last_upper,
    })

common_domain_df = pd.DataFrame(
    common_domain_rows
)

print("\nCOMMON-DOMAIN GRID AUDIT")
print("-" * 92)

print(
    common_domain_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}",
    )
)

# -------------------------------------------------------------------------
# 72-bin common-domain reference
# -------------------------------------------------------------------------

ref_common = common_descriptor_objects[
    72
]["descriptors"]

comparison_rows = []

for n_rad in RADIAL_RESOLUTIONS:

    desc = common_descriptor_objects[
        n_rad
    ]["descriptors"]

    for j, feature in enumerate(
        radial_feature_names
    ):

        x = ref_common[:, j]
        y = desc[:, j]

        rho = float(
            spearmanr(
                x,
                y,
            ).statistic
        )

        abs_change = np.abs(
            y - x
        )

        comparison_rows.append({
            "n_radial_bins": n_rad,
            "feature": feature,
            "spearman_vs_72": rho,
            "median_abs_change":
                float(np.median(abs_change)),
            "mean_abs_change":
                float(np.mean(abs_change)),
            "max_abs_change":
                float(np.max(abs_change)),
        })

common_radial_resolution_df = pd.DataFrame(
    comparison_rows
)

# -------------------------------------------------------------------------
# Rank stability table
# -------------------------------------------------------------------------

print("\nCOMMON-DOMAIN RADIAL-RESOLUTION STABILITY")
print("-" * 92)

rho_table_common = (
    common_radial_resolution_df
    .pivot(
        index="feature",
        columns="n_radial_bins",
        values="spearman_vs_72",
    )
)

print(
    rho_table_common.to_string(
        float_format=lambda x: f"{x:.6f}",
    )
)

# -------------------------------------------------------------------------
# Peak-location comparison
# -------------------------------------------------------------------------

print("\nCOMMON-DOMAIN PEAK-RADIUS STABILITY")
print("-" * 92)

ref_peak = ref_common[:, 6]

peak_rows = []

for n_rad in RADIAL_RESOLUTIONS:

    peak = common_descriptor_objects[
        n_rad
    ]["descriptors"][:, 6]

    rho = float(
        spearmanr(
            ref_peak,
            peak,
        ).statistic
    )

    median_abs = float(
        np.median(
            np.abs(
                peak - ref_peak
            )
        )
    )

    mean_abs = float(
        np.mean(
            np.abs(
                peak - ref_peak
            )
        )
    )

    peak_rows.append({
        "n_radial_bins": n_rad,
        "peak_radius_spearman": rho,
        "median_abs_peak_change":
            median_abs,
        "mean_abs_peak_change":
            mean_abs,
    })

peak_common_df = pd.DataFrame(
    peak_rows
)

print(
    peak_common_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

# -------------------------------------------------------------------------
# Peak-magnitude and integrated-magnitude summaries
# -------------------------------------------------------------------------

print("\nKEY COORDINATE-FREE / GLOBAL FEATURES")
print("-" * 92)

for feature in [
    "integrated_magnitude",
    "radial_centroid",
    "radial_spread",
    "peak_magnitude",
]:

    sub = common_radial_resolution_df[
        common_radial_resolution_df[
            "feature"
        ] == feature
    ]

    print(f"\n{feature}")

    print(
        sub[
            [
                "n_radial_bins",
                "spearman_vs_72",
                "median_abs_change",
                "mean_abs_change",
            ]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}",
        )
    )

# -------------------------------------------------------------------------
# Hard guards
# -------------------------------------------------------------------------

row72 = common_radial_resolution_df[
    common_radial_resolution_df[
        "n_radial_bins"
    ] == 72
]

assert np.allclose(
    row72["spearman_vs_72"],
    1.0,
    atol=1e-12,
)

# Exact expected shell counts
expected_counts = {
    24: 6,
    36: 9,
    72: 18,
}

for n_rad, expected in expected_counts.items():

    observed = int(
        common_domain_df.loc[
            common_domain_df[
                "n_radial_bins"
            ] == n_rad,
            "n_selected_shells",
        ].iloc[0]
    )

    assert observed == expected

print("\nPASS — radial-resolution sensitivity isolated from domain mismatch.")
print(
    "All three resolutions were compared over exactly the same "
    "normalized physical interval."
)

CLO-SKET — PARAMETER SENSITIVITY
CELL 10 — RADIAL RESOLUTION ON EXACT COMMON PHYSICAL DOMAIN

COMMON PHYSICAL DOMAIN
--------------------------------------------------------------------------------------------
Lower edge : 0.083333333
Upper edge : 0.333333333

COMMON-DOMAIN GRID AUDIT
--------------------------------------------------------------------------------------------
 n_radial_bins  n_selected_shells  first_lower_edge  last_upper_edge
            24                  6       0.083333333      0.333333333
            36                  9       0.083333333      0.333333333
            72                 18       0.083333333      0.333333333

COMMON-DOMAIN RADIAL-RESOLUTION STABILITY
--------------------------------------------------------------------------------------------
n_radial_bins              24       36       72
feature                                        
integrated_magnitude 0.942134 0.978486 1.000000
onset_radius         0.463113 0.591605 1.000000
peak_concentratio

In [ ]:
# =============================================================================
# CLO-SKET PARAMETER SENSITIVITY
# CELL 11 — FINAL PARAMETER-SENSITIVITY DIAGNOSTIC LOCK
# =============================================================================

import numpy as np
import pandas as pd

print("=" * 92)
print("CLO-SKET — PARAMETER SENSITIVITY")
print("CELL 11 — FINAL PARAMETER-SENSITIVITY DIAGNOSTIC LOCK")
print("=" * 92)

# -------------------------------------------------------------------------
# 1. Threshold sensitivity summary
# -------------------------------------------------------------------------

threshold_variants = [
    "threshold_0.05",
    "threshold_0.15",
]

threshold_rows = []

for variant in threshold_variants:

    sub = sensitivity_feature_df[
        sensitivity_feature_df["variant"] == variant
    ].copy()

    for feature in radial_feature_names:

        row = sub[
            sub["feature"] == feature
        ].iloc[0]

        threshold_rows.append({
            "variant": variant,
            "feature": feature,
            "spearman_rho": float(
                row["spearman_rho"]
            ),
            "unchanged_prop": float(
                row["unchanged_prop"]
            ),
        })

threshold_lock_df = pd.DataFrame(
    threshold_rows
)

# -------------------------------------------------------------------------
# 2. Concentration-width sensitivity summary
# -------------------------------------------------------------------------

concentration_variants = [
    "concentration_2.0",
    "concentration_6.0",
]

concentration_rows = []

for variant in concentration_variants:

    sub = sensitivity_feature_df[
        sensitivity_feature_df["variant"] == variant
    ].copy()

    for feature in radial_feature_names:

        row = sub[
            sub["feature"] == feature
        ].iloc[0]

        concentration_rows.append({
            "variant": variant,
            "feature": feature,
            "spearman_rho": float(
                row["spearman_rho"]
            ),
            "median_abs_change": float(
                row["median_abs_change"]
            ),
            "unchanged_prop": float(
                row["unchanged_prop"]
            ),
        })

concentration_lock_df = pd.DataFrame(
    concentration_rows
)

# -------------------------------------------------------------------------
# 3. Angular-resolution summary
# -------------------------------------------------------------------------

angular_lock_df = angular_resolution_df.copy()

# -------------------------------------------------------------------------
# 4. Radial-domain summary
# -------------------------------------------------------------------------

radial_domain_lock_df = (
    full_domain_summary_df.copy()
)

# -------------------------------------------------------------------------
# 5. Common-domain radial-resolution summary
# -------------------------------------------------------------------------

radial_resolution_lock_df = (
    common_radial_resolution_df.copy()
)

# -------------------------------------------------------------------------
# 6. Compact manuscript-facing feature classification
# -------------------------------------------------------------------------

feature_classification = pd.DataFrame([
    {
        "feature_family":
            "Integrated magnitude",
        "sensitivity_status":
            "Substantially robust",
        "evidence":
            "High rank stability under domain and radial-resolution perturbation",
    },
    {
        "feature_family":
            "Radial centroid",
        "sensitivity_status":
            "Substantially robust",
        "evidence":
            "High rank stability under domain and radial-resolution perturbation",
    },
    {
        "feature_family":
            "Radial spread",
        "sensitivity_status":
            "Moderately to substantially robust",
        "evidence":
            "Good stability; some degradation at coarse radial resolution",
    },
    {
        "feature_family":
            "Peak magnitude",
        "sensitivity_status":
            "Moderately robust",
        "evidence":
            "High angular-resolution stability; some radial-domain/resolution dependence",
    },
    {
        "feature_family":
            "Peak concentration",
        "sensitivity_status":
            "Parameter-sensitive",
        "evidence":
            "Depends directly on concentration width and radial discretization/domain",
    },
    {
        "feature_family":
            "Onset radius",
        "sensitivity_status":
            "Parameter-sensitive",
        "evidence":
            "Sensitive to support threshold, domain boundaries, and radial resolution",
    },
    {
        "feature_family":
            "Termination radius",
        "sensitivity_status":
            "Parameter-sensitive",
        "evidence":
            "Sensitive to support threshold, domain boundaries, and radial resolution",
    },
    {
        "feature_family":
            "Peak radius",
        "sensitivity_status":
            "Moderately sensitive",
        "evidence":
            "Stable to threshold/angular resolution but sensitive to radial domain/resolution",
    },
])

# -------------------------------------------------------------------------
# 7. Key numerical locks
# -------------------------------------------------------------------------

# Angular discretization
row36_ang = angular_resolution_df[
    angular_resolution_df[
        "n_angular_bins"
    ] == 36
].iloc[0]

row24_ang = angular_resolution_df[
    angular_resolution_df[
        "n_angular_bins"
    ] == 24
].iloc[0]

# Radial resolution
peak36 = peak_common_df[
    peak_common_df[
        "n_radial_bins"
    ] == 36
].iloc[0]

peak24 = peak_common_df[
    peak_common_df[
        "n_radial_bins"
    ] == 24
].iloc[0]

# Primary radial boundary
primary_domain_row = (
    full_domain_summary_df[
        full_domain_summary_df[
            "label"
        ] == "primary"
    ].iloc[0]
)

# Outward-most tested domain
outward3_row = (
    full_domain_summary_df[
        full_domain_summary_df[
            "label"
        ] == "outward_3"
    ].iloc[0]
)

# Primary-upper-boundary subset
primary_peak_r = primary_full_desc[:, 6]
primary_upper_mask = (
    primary_peak_r == 27.5
)

outward3_peak_r = (
    domain_objects_full[
        "outward_3"
    ]["descriptors"][:, 6]
)

upper_boundary_move_beyond = float(
    np.mean(
        outward3_peak_r[
            primary_upper_mask
        ]
        > 27.5
    )
)

# -------------------------------------------------------------------------
# 8. Manuscript-facing summary table
# -------------------------------------------------------------------------

parameter_sensitivity_summary = pd.DataFrame([
    {
        "analysis":
            "Support threshold",
        "primary_setting":
            "0.10",
        "tested_alternatives":
            "0.05, 0.15",
        "main_result":
            (
                "Six of eight radial descriptors unchanged exactly; "
                "onset/termination changed in a small minority of sketches"
            ),
        "interpretation":
            "Strong robustness except threshold-defined support endpoints",
    },
    {
        "analysis":
            "Concentration half-width",
        "primary_setting":
            "±4 radial units",
        "tested_alternatives":
            "±2, ±6",
        "main_result":
            (
                "Only concentration changed; all seven other "
                "descriptors were identical"
            ),
        "interpretation":
            "Expected parameter dependence confined to concentration feature",
    },
    {
        "analysis":
            "Angular resolution",
        "primary_setting":
            "72 bins",
        "tested_alternatives":
            "36, 24 bins",
        "main_result":
            (
                f"R2 rank correlation vs 72 bins: "
                f"{row36_ang['R2_spearman']:.3f} at 36, "
                f"{row24_ang['R2_spearman']:.3f} at 24"
            ),
        "interpretation":
            "Second-harmonic magnitude highly robust to angular discretization",
    },
    {
        "analysis":
            "Radial domain",
        "primary_setting":
            "3.5–27.5",
        "tested_alternatives":
            "5.5–25.5 through 0.5–30.5",
        "main_result":
            (
                f"Primary endpoint-peak proportion "
                f"{primary_domain_row['either_endpoint_prop']:.3f}; "
                f"widest-domain endpoint proportion "
                f"{outward3_row['either_endpoint_prop']:.3f}"
            ),
        "interpretation":
            "Localized radial descriptors depend materially on the analysis window",
    },
    {
        "analysis":
            "Upper-boundary censoring",
        "primary_setting":
            "27.5 upper boundary",
        "tested_alternatives":
            "expanded to 30.5",
        "main_result":
            (
                f"{upper_boundary_move_beyond:.3f} of primary "
                f"27.5 peaks moved beyond 27.5"
            ),
        "interpretation":
            "A substantial subset of upper-boundary peaks is domain-censored",
    },
    {
        "analysis":
            "Radial resolution",
        "primary_setting":
            "72 bins",
        "tested_alternatives":
            "36, 24 bins on exact common physical domain",
        "main_result":
            (
                f"Peak-radius rho vs 72: "
                f"{peak36['peak_radius_spearman']:.3f} at 36, "
                f"{peak24['peak_radius_spearman']:.3f} at 24"
            ),
        "interpretation":
            "Global radial summaries are more stable than localized discrete descriptors",
    },
])

# -------------------------------------------------------------------------
# 9. Print locked tables
# -------------------------------------------------------------------------

print("\nMANUSCRIPT-FACING PARAMETER-SENSITIVITY SUMMARY")
print("-" * 92)

print(
    parameter_sensitivity_summary.to_string(
        index=False
    )
)

print("\nFEATURE-LEVEL INTERPRETATION")
print("-" * 92)

print(
    feature_classification.to_string(
        index=False
    )
)

# -------------------------------------------------------------------------
# 10. Hard scientific guards
# -------------------------------------------------------------------------

# Angular magnitude must remain strongly stable at both tested coarser resolutions
assert (
    row36_ang["R2_spearman"]
    > 0.99
)

assert (
    row24_ang["R2_spearman"]
    > 0.99
)

# Threshold sensitivity must leave peak radius and magnitude exactly unchanged
for variant in threshold_variants:

    arr = sensitivity_objects[
        variant
    ]

    assert np.array_equal(
        arr[:, 6],
        primary[:, 6],
    )

    assert np.array_equal(
        arr[:, 7],
        primary[:, 7],
    )

# Concentration-width sensitivity must not alter the other seven descriptors
for variant in concentration_variants:

    arr = sensitivity_objects[
        variant
    ]

    non_concentration_cols = [
        0, 1, 2, 4, 5, 6, 7
    ]

    assert np.allclose(
        arr[:, non_concentration_cols],
        primary[:, non_concentration_cols],
        atol=0.0,
        rtol=0.0,
    )

# Primary domain boundary issue must be acknowledged
assert (
    primary_domain_row[
        "either_endpoint_prop"
    ]
    > 0.20
)

assert upper_boundary_move_beyond > 0.30

# -------------------------------------------------------------------------
# 11. Final interpretation lock
# -------------------------------------------------------------------------

print("\n" + "=" * 92)
print("FINAL PARAMETER-SENSITIVITY INTERPRETATION LOCK")
print("=" * 92)

print("""
1. SUPPORT THRESHOLD
   Varying the support threshold from 0.10 to 0.05 or 0.15 leaves
   six of eight radial descriptors exactly unchanged. Sensitivity is
   largely confined to the threshold-defined onset and termination radii.

2. CONCENTRATION WIDTH
   Varying the concentration half-width changes the concentration
   coordinate by construction but leaves the remaining seven radial
   descriptors unchanged.

3. ANGULAR DISCRETIZATION
   The second-harmonic magnitude field is highly stable when the
   canonical 72 angular bins are coarsened to 36 or 24 bins.
   Therefore the principal second-harmonic magnitude structure is not
   an artifact of the exact 72-bin angular discretization.

4. RADIAL DOMAIN
   The primary 3.5–27.5 window materially affects localized radial
   descriptors. Approximately 22% of primary peak radii occur at a
   domain endpoint, and a substantial fraction of peaks at the upper
   boundary migrate beyond 27.5 when the domain is expanded.

5. RADIAL RESOLUTION
   Integrated magnitude, centroid and spread remain comparatively
   stable under coarser radial resolutions. Peak radius, concentration,
   onset and termination are more sensitive to radial discretization.

6. CLAIM BOUNDARY
   The frozen primary parameters are not claimed to be optimal or
   universal. Sensitivity analyses support the robustness of broad
   second-harmonic and global radial structure while requiring more
   cautious interpretation of localized peak- and support-based
   descriptors.

7. PRIMARY ANALYSIS STATUS
   No primary parameter was changed after sensitivity analysis.
   The original 14-D representation remains the frozen primary
   specification; sensitivity results constrain interpretation rather
   than selecting a new post-hoc configuration.
""")

print("PASS — Point 3 parameter sensitivity scientifically locked.")

CLO-SKET — PARAMETER SENSITIVITY
CELL 11 — FINAL PARAMETER-SENSITIVITY DIAGNOSTIC LOCK

MANUSCRIPT-FACING PARAMETER-SENSITIVITY SUMMARY
--------------------------------------------------------------------------------------------
                analysis     primary_setting                         tested_alternatives                                                                                                  main_result                                                              interpretation
       Support threshold                0.10                                  0.05, 0.15 Six of eight radial descriptors unchanged exactly; onset/termination changed in a small minority of sketches                Strong robustness except threshold-defined support endpoints
Concentration half-width     ±4 radial units                                      ±2, ±6                                       Only concentration changed; all seven other descriptors were identical             Expected para